## Importing libraries

In [ ]:
import mne
from pathlib import Path
%matplotlib qt

## Loading the data

In [ ]:
base_dir = Path.cwd().parent
subject_dir = base_dir / "data" / "raw" / "V00test_data"   # Change directory
input_fname = subject_dir / "com_ruido.bdf"                # Change filename

raw = mne.io.read_raw_bdf(
    input_fname,
    eog=None,
    misc=None,
    stim_channel='auto',
    exclude=(),
    preload=True,
)

## Verify data structure

In [ ]:
print(raw.info)
print('')
print(f'Channels: {len(raw.ch_names)}: {raw.ch_names}')
print('')
print(raw.annotations)

## Set EOG and EMG channels

In [ ]:
raw.set_channel_types({
    'EOG': 'eog',
    'EMG': 'emg',
})

raw.set_montage('standard_1020')

## Plot raw data

In [ ]:
raw.plot(duration=5, n_channels=30, scalings='auto', block=False);

In [ ]:
raw.compute_psd().plot();

In [ ]:
raw.plot_sensors(show_names=True);

In [ ]:
raw.compute_psd(fmin=1, fmax=40, method='welch').plot_topo();

# Importing epochs

In [ ]:
epochs = mne.read_epochs('/home/victormoraes/Documents/GitHub/TMS-EEG_ContextTree_Processing_and_Analysis/data/processed/V00test/emg_processed/V00test_emg_epochs_processed.fif')


In [ ]:
epochs.plot(picks='emg')

In [ ]:
print(epochs.annotations)

# Importing fif file

In [ ]:
import mne
import numpy as np

# ---- 1. Carrega raw e extrai eventos das annotations ----
raw = mne.io.read_raw_fif(
    '/home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_raw_processed.fif',
    preload=False
)

events, event_id = mne.events_from_annotations(raw)

# manter apenas 8Bit
keep_desc = ['8Bit 1', '8Bit 2', '8Bit 3']
keep_codes = [event_id[d] for d in keep_desc]
mask_8bit = np.isin(events[:, 2], keep_codes)
events_8bit = events[mask_8bit]

print(f"Total de eventos 8Bit na sequência original: {len(events_8bit)}")
print(f"Primeiros 20 códigos: {events_8bit[:20, 2]}")
print(f"event_id original: {event_id}")


In [ ]:
# ---- 2. Carrega epochs ----
epochs = mne.read_epochs(
    '/home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_epochs_processed.fif',
    preload=False
)

print(f"Total de epochs sobreviventes: {len(epochs)}")
print(f"event_id das epochs: {epochs.event_id}")
print(f"Primeiros 20 selection: {epochs.selection[:20]}")
print(f"Tamanho de selection: {len(epochs.selection)}")
print(f"Primeiros 20 eventos das epochs: {epochs.events[:20, 2]}")


In [ ]:
# ---- 3. Verifica se selection indexa a sequência original 8Bit ----
for i in range(min(20, len(epochs))):
    idx_original = epochs.selection[i]
    original_code = events_8bit[idx_original, 2] if idx_original < len(events_8bit) else None
    epoch_code = epochs.events[i, 2]

    print(
        f"epoch {i:02d} | selection={idx_original:03d} | "
        f"epoch_code={epoch_code} | original_code={original_code}"
    )


In [ ]:
print(f"Total eventos 8Bit na sequência original: {len(events_8bit)}")
print(f"Total epochs sobreviventes: {len(epochs)}")
print(f"Epochs rejeitadas: {len(events_8bit) - len(epochs)}")

# Ver onde estão os "buracos" (epochs rejeitadas)
todos = set(range(len(events_8bit)))
sobreviventes = set(epochs.selection)
rejeitadas = sorted(todos - sobreviventes)

print(f"\nÍndices rejeitados (primeiros 30): {rejeitadas[:30]}")
print(f"Primeiro rejeitado: {rejeitadas[0] if rejeitadas else 'nenhum'}")


In [ ]:
from collections import Counter

code_to_name = {1: '8Bit1', 2: '8Bit2', 3: '8Bit3'}

contexts = []
for i in range(len(epochs)):
    idx = epochs.selection[i]
    atual = events_8bit[idx, 2]
    
    if idx == 0:
        contexto = f"inicio->{code_to_name[atual]}"
    else:
        anterior = events_8bit[idx - 1, 2]
        contexto = f"{code_to_name[anterior]}->{code_to_name[atual]}"
    
    contexts.append(contexto)

contagem = Counter(contexts)

print(f"Total de epochs com contexto: {len(contexts)}")
print(f"\nContagem por contexto:")
for ctx, n in sorted(contagem.items()):
    print(f"  {ctx}: {n}")


In [ ]:
label_map = {'8Bit1': '0', '8Bit2': '1', '8Bit3': '2', 'inicio': 'inicio'}

new_contexts = []
for ctx in epochs.metadata['context']:
    parts = ctx.split('->')
    mapped = [label_map[p] for p in parts]
    new_contexts.append(''.join(mapped))

epochs.metadata['context'] = new_contexts
print(epochs.metadata['context'].value_counts())


In [3]:
import mne
raw = mne.io.read_raw_fif(
    "/home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_raw_processed.fif",
    preload=False
)
events, event_id = mne.events_from_annotations(raw)
print(event_id)

Opening raw data file /home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_raw_processed.fif...
    Range : 0 ... 15779999 =      0.000 ...   789.000 secs
Ready.
Opening raw data file /home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_raw_processed-1.fif...
    Range : 15780000 ... 31559999 =    789.000 ...  1578.000 secs
Ready.
Opening raw data file /home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_raw_processed-2.fif...
    Range : 31560000 ... 47339999 =   1578.000 ...  2367.000 secs
Ready.
Opening raw data file /home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_raw_processed-3.fif...
    Range : 47340000 ... 61829999 =   2367.000 ...  3091.500 secs
Ready.


/tmp/ipykernel_71458/2949118422.py:2: RuntimeWarning: This filename (/home/victormoraes/Documents/GitHub/MEPs-and-TEPs-with-Context-Tree/data/processed/V02/processed/V02_raw_processed.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(


Used Annotations descriptions: [np.str_('8Bit 1'), np.str_('8Bit 2'), np.str_('8Bit 3'), np.str_('Unk SW')]
{np.str_('8Bit 1'): 1, np.str_('8Bit 2'): 2, np.str_('8Bit 3'): 3, np.str_('Unk SW'): 4}
